# 03 · Interpretation, stability & sensitivity (T08–T10)

Reads the authoritative run outputs after `--stage stability compare sensitivity`. Selection was frozen BEFORE any season/weather view; this notebook only interprets. Core logic lives in `src/air_quality_clustering/stability.py` and `discovery.py`.

In [ ]:
import json
import sys
from pathlib import Path

import pandas as pd
from IPython.display import Image, display

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))

def latest_run(stage):
    runs = []
    for d in (ROOT / 'results' / 'runs').iterdir():
        mp = d / 'manifest.json'
        if not mp.is_file():
            continue
        m = json.loads(mp.read_text(encoding='utf-8'))
        done = m.get('stages_completed') or m.get('stage_summary', {}).get('stages_completed', [])
        if stage in done and m.get('status') in ('completed', 'completed_with_warnings'):
            runs.append((d.name, d, m))
    if not runs:
        raise RuntimeError(f'no completed run contains stage {stage}; run scripts/run_project.py first')
    return max(runs, key=lambda t: t[0])

run_name, RUN, manifest = latest_run('sensitivity')
print('run_id:', run_name)
selection = json.loads((RUN / 'selection.json').read_text(encoding='utf-8'))
for a in ('kmeans', 'ward'):
    r = selection['methods'][a]['representative']
    print(a, 'representative:', r)
print('dbscan:', selection['dbscan']['representative'], selection['dbscan']['reason'])

## 1. Temporal-block stability (20 refit resamples per shortlisted k)

Failed repetitions are counted, not masked. These ARIs carry exploratory selection bias because k was chosen with them; they are not confidence intervals.

In [ ]:
pd.read_csv(RUN / 'stability' / 'aggregate.csv').round(4)

In [ ]:
display(Image(filename=str(RUN / 'figures' / 'F12_stability_sensitivity.png')))

## 2. Cluster profiles in ORIGINAL units

Medians/IQRs of raw daily concentrations by actual label; display order by PM2.5 median (mapping in artifacts/display_mapping.json). No back-transformed log centroids.

In [ ]:
profiles = pd.read_csv(RUN / 'tables' / 'cluster_profiles.csv')
profiles[profiles.algorithm == 'kmeans'].pivot(index='cluster', columns='pollutant', values='median').round(1)

In [ ]:
display(Image(filename=str(RUN / 'figures' / 'F09_profiles_kmeans.png')))

In [ ]:
display(Image(filename=str(RUN / 'figures' / 'F09_profiles_ward.png')))

## 3. Method agreement

Contingency + representative ARI; same-k ARI separates geometry from cluster count.

In [ ]:
print(pd.read_csv(RUN / 'tables' / 'contingency.csv', index_col=0))
pd.read_csv(latest_run('cluster')[1] / 'tables' / 'same_k_ari.csv').round(4)

In [ ]:
display(Image(filename=str(RUN / 'figures' / 'F08_pca_panels.png')))

In [ ]:
display(Image(filename=str(RUN / 'figures' / 'F10_agreement.png')))

## 4. Background AFTER freezing (descriptive only)

Every proportion states its denominator. No causal claims; clusters are not sources or health grades.

In [ ]:
pd.read_csv(RUN / 'tables' / 'season_pct_of_cluster_kmeans.csv', index_col=0).round(3)

In [ ]:
display(Image(filename=str(RUN / 'figures' / 'F11_background.png')))

## 5. Transformation sensitivity

raw+StandardScaler at frozen k vs the main log1p representation.

In [ ]:
pd.read_csv(RUN / 'sensitivity' / 'representation_ari.csv').round(4)

## 6. Report

The full English report generated from this run's numbers: `report.md` in the run directory (stage export).